In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import pandas as pd
import seaborn as sns

sns.set_style("whitegrid")

def _half_violin(ax, data, pos, widths=0.75, side="left",
                 facecolor="tab:blue", edgecolor="black", alpha=0.35):
    v = ax.violinplot([data], positions=[pos], widths=widths,
                      showmeans=False, showmedians=False, showextrema=False)
    body = v["bodies"][0]
    path = body.get_paths()[0]
    verts = path.vertices
    x_mean = np.mean(verts[:, 0])

    if side == "left":
        verts[:, 0] = np.clip(verts[:, 0], np.min(verts[:, 0]), x_mean)
    else:
        verts[:, 0] = np.clip(verts[:, 0], x_mean, np.max(verts[:, 0]))

    body.set_facecolor(facecolor)
    body.set_edgecolor(edgecolor)
    body.set_alpha(alpha)


def plot_task_box_violin_scatter(
    dfs,
    task_labels,
    metric_col,
    title,
    color,
    save_path,
    jitter_strength=0.1,
    scatter_shift=+0.20,
    box_width=0.22,
    violin_width=0.75,
    violin_side="left",
    point_size=80,
    point_alpha=0.35,
    angle=0,
    figsize=(10, 4)
):
    data = [df[metric_col].dropna().values for df in dfs]
    positions = np.arange(1, len(dfs) + 1)

    fig, ax = plt.subplots(figsize=figsize)

    # Scatter
    for pos, d in zip(positions, data):
        jitter = np.random.uniform(-jitter_strength, jitter_strength, size=len(d))
        ax.scatter(
            pos + scatter_shift + jitter, d,
            color=color, alpha=point_alpha,
            edgecolor="black", s=point_size
        )

    # Boxplot
    box = ax.boxplot(data, positions=positions,
                     widths=box_width, patch_artist=True)
    for b in box["boxes"]:
        b.set_facecolor(color)
        b.set_edgecolor("black")
    for m in box["medians"]:
        m.set_color("black")

    # Half violin
    for pos, d in zip(positions, data):
        _half_violin(
            ax, d, pos,
            widths=violin_width,
            side=violin_side,
            facecolor=color,
            edgecolor="black",
            alpha=0.35
        )

    # Axes formatting
    ax.set_xticks(positions)
    ax.set_xticklabels(task_labels, fontsize=14, rotation=angle)
    ax.grid(True, linestyle="--", alpha=0.6)

    ax.yaxis.set_major_formatter(
        FuncFormatter(lambda v, _: f"{v:.3f}")
    )
    for label in ax.get_yticklabels():
        label.set_fontsize(12)

    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_facecolor("white")
    fig.patch.set_alpha(0.0)

    plt.tight_layout()
    plt.savefig(save_path, dpi=400, bbox_inches="tight")
    plt.close(fig)


In [5]:
df_binary = pd.read_csv("metrics_binary.csv")
df_multiclass = pd.read_csv("metrics_multiclass.csv")
df_regression = pd.read_csv("metrics_regression.csv")
df_survival = pd.read_csv("metrics_survival.csv")



metric_colors = {
    "act_corr_mean": "#B98F8F",
    "act_cosine_mean":   "#B98F8F",  
    "act_r2": "#B98F8F",  
}

dfs = [df_binary, df_multiclass, df_regression, df_survival]

task_labels = [
    "Binary",
    "Multiclass",
    "Regression",
    "Survival"
]


metrics = [
    "act_corr_mean",
    "act_cosine_mean",
    "act_r2"
]


output_dir = "./Figures_Metrics/" 

metric_titles = {
    "act_corr_mean": "Pearson correlation coefficient",
    "act_cosine_mean" : "Cosine similarity",
    "act_r2" : "Coefficient of determination",
    "relL2_mean": "Relative L2 error",
    "cindex": "C-index"
}



In [6]:
for metric in metrics:
    color = metric_colors.get(metric, "#333333")  
    title = metric_titles.get(metric, metric)
    save_path = f"{output_dir}{metric}_4tasks.png"

    plot_task_box_violin_scatter(
        dfs=dfs,
        task_labels=task_labels,
        metric_col=metric,
        title=title,
        color=color,
        save_path=save_path,
        violin_side="left",
        figsize=(8, 6)
    )
